# **AJUSTES INICIAIS**

In [ ]:
import pandas as pd
import numpy as np

In [14]:
df_parana = pd.read_excel("cepea-parana-01_16-06_26.xls")
df_paranagua = pd.read_excel("cepea-paranagua-01_16-06_26.xls")

In [15]:
df_parana.head()

,Data,À vista R$,À vista US$
0,04/01/2016,"78,14","19,30"
1,05/01/2016,"79,00","19,79"
2,06/01/2016,"78,55","19,55"
3,07/01/2016,"78,82","19,49"
4,08/01/2016,"78,96","19,57"


In [16]:
df_paranagua.head()

,Data,À vista R$,À vista US$
0,04/01/2016,"81,64","20,17"
1,05/01/2016,"82,26","20,61"
2,06/01/2016,"81,84","20,37"
3,07/01/2016,"82,35","20,36"
4,08/01/2016,"83,16","20,61"


In [17]:
df_parana["indicador"] = "parana"
df_paranagua["indicador"] = "paranagua"

df_daily = pd.concat([df_parana, df_paranagua], ignore_index=True)

In [18]:
df_daily.head()

,Data,À vista R$,À vista US$,indicador
0,04/01/2016,"78,14","19,30",parana
1,05/01/2016,"79,00","19,79",parana
2,06/01/2016,"78,55","19,55",parana
3,07/01/2016,"78,82","19,49",parana
4,08/01/2016,"78,96","19,57",parana


In [20]:
df_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5194 entries, 0 to 5193
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Data         5194 non-null   object
 1   À vista R$   5194 non-null   object
 2   À vista US$  5194 non-null   object
 3   indicador    5194 non-null   object
dtypes: object(4)
memory usage: 162.4+ KB


In [24]:
df_daily['indicador'].unique()

array(['parana', 'paranagua'], dtype=object)

In [25]:
df_daily.isna().sum()

,0
Data,0
À vista R$,0
À vista US$,0
indicador,0


In [26]:
df_daily.isnull().sum()

,0
Data,0
À vista R$,0
À vista US$,0
indicador,0


In [28]:
df_daily['Data'] = pd.to_datetime(df_daily['Data'], format='%d/%m/%Y')

In [29]:
df_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5194 entries, 0 to 5193
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Data         5194 non-null   datetime64[ns]
 1   À vista R$   5194 non-null   object        
 2   À vista US$  5194 non-null   object        
 3   indicador    5194 non-null   object        
dtypes: datetime64[ns](1), object(3)
memory usage: 162.4+ KB


In [31]:
df_daily['À vista R$'] = df_daily['À vista R$'].str.replace(',', '.')
df_daily['À vista US$'] = df_daily['À vista US$'].str.replace(',', '.')
df_daily.head()

,Data,À vista R$,À vista US$,indicador
0,2016-01-04,78.14,19.30,parana
1,2016-01-05,79.00,19.79,parana
2,2016-01-06,78.55,19.55,parana
3,2016-01-07,78.82,19.49,parana
4,2016-01-08,78.96,19.57,parana


In [34]:
df_daily['À vista R$'] = df_daily['À vista R$'].astype(float)
df_daily['À vista US$'] = df_daily['À vista US$'].astype(float)
df_daily = df_daily.sort_values(by='Data').reset_index(drop=True)
df_daily.head()

,Data,À vista R$,À vista US$,indicador
0,2016-01-04,78.14,19.30,parana
1,2016-01-04,81.64,20.17,paranagua
2,2016-01-05,82.26,20.61,paranagua
3,2016-01-05,79.00,19.79,parana
4,2016-01-06,78.55,19.55,parana


In [38]:
df_daily = df_daily.drop_duplicates()
print(f'Duplicatas restantes: {df_daily.duplicated().sum()}')

Duplicatas restantes: 0


In [37]:
invalid_prices = df_daily[(df_daily['À vista R$'] <= 0) | (df_daily['À vista US$'] <= 0)]
invalid_prices

,Data,À vista R$,À vista US$,indicador


## Exportando dataset diario

In [39]:
df_daily.to_parquet('df_daily.parquet', index=False)

## Dataset semanal

In [40]:
df_daily.head()

,Data,À vista R$,À vista US$,indicador
0,2016-01-04,78.14,19.30,parana
1,2016-01-04,81.64,20.17,paranagua
2,2016-01-05,82.26,20.61,paranagua
3,2016-01-05,79.00,19.79,parana
4,2016-01-06,78.55,19.55,parana


In [44]:
df_weekly = (
    df_daily
    .sort_values(["indicador", "Data"])
    .set_index("Data")
    .groupby("indicador")
    .resample("W-FRI")
    .agg(
        preco_rs_medio=("À vista R$", "mean"),
        preco_rs_fechamento=("À vista R$", "last"),
        preco_rs_max=("À vista R$", "max"),
        preco_rs_min=("À vista R$", "min"),
        preco_usd_medio=("À vista US$", "mean"),
        preco_usd_fechamento=("À vista US$", "last"),
        preco_usd_max=("À vista US$", "max"),
        preco_usd_min=("À vista US$", "min"),
        qtd_dias_semana=("À vista R$", "count"),
    )
    .reset_index()
    .rename(columns={"Data": "semana"})
)

df_weekly["retorno_rs_1w"] = (
    df_weekly
    .sort_values(["indicador", "semana"])
    .groupby("indicador")["preco_rs_fechamento"]
    .pct_change()
)

df_weekly["retorno_usd_1w"] = (
    df_weekly
    .sort_values(["indicador", "semana"])
    .groupby("indicador")["preco_usd_fechamento"]
    .pct_change()
)

df_weekly.head()

,indicador,semana,preco_rs_medio,preco_rs_fechamento,preco_rs_max,preco_rs_min,preco_usd_medio,preco_usd_fechamento,preco_usd_max,preco_usd_min,qtd_dias_semana,retorno_rs_1w,retorno_usd_1w
0,parana,2016-01-08,78.694,78.96,79.00,78.14,19.540,19.57,19.79,19.30,5,NaN,NaN
1,parana,2016-01-15,80.298,79.79,80.76,79.79,19.940,19.70,20.22,19.70,5,0.010512,0.006643
2,parana,2016-01-22,78.294,77.81,79.55,77.81,19.144,18.95,19.74,18.78,5,-0.024815,-0.038071
3,parana,2016-01-29,76.572,75.64,77.28,75.64,18.812,18.79,18.92,18.67,5,-0.027888,-0.008443
4,parana,2016-02-05,74.208,72.37,75.29,72.37,18.856,18.49,19.12,18.49,5,-0.043231,-0.015966


In [45]:
df_weekly.dropna(inplace=True)
df_weekly = df_weekly.sort_values(by='semana').reset_index(drop=True)
df_weekly.head()

,indicador,semana,preco_rs_medio,preco_rs_fechamento,preco_rs_max,preco_rs_min,preco_usd_medio,preco_usd_fechamento,preco_usd_max,preco_usd_min,qtd_dias_semana,retorno_rs_1w,retorno_usd_1w
0,parana,2016-01-15,80.298,79.79,80.76,79.79,19.940,19.70,20.22,19.70,5,0.010512,0.006643
1,paranagua,2016-01-15,84.822,84.42,85.70,83.35,21.066,20.85,21.39,20.59,5,0.015152,0.011645
2,paranagua,2016-01-22,83.220,82.94,83.98,82.39,20.344,20.19,20.83,20.02,5,-0.017531,-0.031655
3,parana,2016-01-22,78.294,77.81,79.55,77.81,19.144,18.95,19.74,18.78,5,-0.024815,-0.038071
4,parana,2016-01-29,76.572,75.64,77.28,75.64,18.812,18.79,18.92,18.67,5,-0.027888,-0.008443


In [46]:
df_weekly.to_parquet('df_weekly.parquet', index=False)